# PICS QF: reported measurements linked to MEROPS-position windows

Extract the authors' coefficients and relative activities with their enzyme, exact substrate chemistry, cut window and source cell. This notebook produces in-memory measurement tables for later integration.

## Definitions

Sources: `raw/mmp_pics/mmc8.xlsx`, primary-paper Figure 6 and QF methods in `Activesite.pdf`, DOI `10.1016/j.matbio.2015.09.003`. `measurements` contains one row per reported `source_k` coefficient or merged relative activity. Per-round relative percentages and raw signals also remain available in `qf`.

The window columns are P4, P3, P2, P1, P1p, P2p, P3p, P4p. P4 is unavailable; Dpa remains its original nonstandard residue at P2p. Mca and terminal amidation are retained separately. The source coefficient keeps its original value with `unit=None` because its column does not declare a calibrated unit; relative activity has unit `percent_of_PLG_reference`. Both are measurements. No rate is fitted, no activity threshold applied, and no CSV written.

In [1]:
from pathlib import Path
import sys
import logging

import numpy as np
import pandas as pd
from IPython.display import display

ROOT = next(p for p in (Path.cwd().resolve(), *Path.cwd().resolve().parents)
            if (p / "assets/tools/proteolysis_data_manager").is_dir())
sys.path.insert(0, str(ROOT / "assets/tools/proteolysis_data_manager/src"))
RAW = ROOT / "data/proteolysis/window_neighbourhood_datasets/raw"
pd.set_option("display.max_columns", None)
logging.getLogger("proteolysis_data_manager.analysis.window_neighbourhood.qpisa").setLevel(logging.WARNING)
logging.getLogger("proteolysis_data_manager.analysis.window_neighbourhood.pics").setLevel(logging.WARNING)

from proteolysis_data_manager.analysis.window_neighbourhood.pics import read_qf_assay, qf_measurements_with_windows
qf = read_qf_assay(RAW / "mmp_pics/mmc8.xlsx")
measurements = qf_measurements_with_windows(qf)

In [2]:
overview = measurements.groupby(["measurement_type", "unit_status"], dropna=False).agg(
    measurement_rows=("value", "size"), enzymes=("enzyme", "nunique"),
    substrates=("substrate", "nunique"), windows=("window", "nunique"),
).reset_index()
display(overview)
assert measurements["source_record_id"].is_unique
assert measurements["P4"].isna().all() and measurements["P2p"].eq("Dpa").all()
assert np.isfinite(measurements["value"]).all()

,measurement_type,unit_status,measurement_rows,enzymes,substrates,windows
0,relative_activity,reported,54,9,6,6
1,source_k,not_reported,162,9,6,6


## Inspect the measurement records

Change the explicit filters below. `selected_measurements` retains all fields, including eight positional tokens, source cell, publication, pH 7.5, temperature 37 °C, 1 µM substrate, and the reported 1–10 nM enzyme range. The range is not replaced by a guessed exact concentration. Coefficient records retain their analysis round and t1/t2 interval; activity summaries retain the source SD.

In [3]:
ENZYME = "MMP1"
PREVIEW_ROWS_PER_TYPE = 6

selected_measurements = measurements.loc[measurements["enzyme"].eq(ENZYME)].copy()
columns = ["enzyme", "substrate", "window", "measurement_type", "value", "unit", "analysis_round", "value_sd", "source_value_cell"]
display(selected_measurements.groupby("measurement_type", sort=False).head(PREVIEW_ROWS_PER_TYPE)[columns].reset_index(drop=True))

,enzyme,substrate,window,measurement_type,value,unit,analysis_round,value_sd,source_value_cell
0,MMP1,PLG↓L,- P L G | L Dpa A R,source_k,1.128738,None,1.0,NaN,L7
1,MMP1,PAN↓L,- P A N | L Dpa A R,source_k,0.673630,None,1.0,NaN,L8
2,MMP1,PLN↓L,- P L N | L Dpa A R,source_k,2.851020,None,1.0,NaN,L9
3,MMP1,PAG↓L,- P A G | L Dpa A R,source_k,0.121609,None,1.0,NaN,L10
4,MMP1,VLG↓L,- V L G | L Dpa A R,source_k,0.207615,None,1.0,NaN,L11
5,MMP1,VAN↓L,- V A N | L Dpa A R,source_k,0.803788,None,1.0,NaN,L12
6,MMP1,PLG↓L,- P L G | L Dpa A R,relative_activity,100.000000,percent_of_PLG_reference,NaN,0.000000,O7
7,MMP1,PAN↓L,- P A N | L Dpa A R,relative_activity,61.819773,percent_of_PLG_reference,NaN,3.706392,O8
8,MMP1,PLN↓L,- P L N | L Dpa A R,relative_activity,249.094666,percent_of_PLG_reference,NaN,6.045053,O9
9,MMP1,PAG↓L,- P A G | L Dpa A R,relative_activity,11.076649,percent_of_PLG_reference,NaN,0.524453,O10


### Remarks

The QF assay covers 54 enzyme–substrate pairs; it does not assign kinetics to every proteomic PICS window. The merged source means and SD are checked against the three reported analysis-round percentages. Round identities and measured-well labels are retained; these should not be collapsed into an inflated number of independent substrates.

### Notes

## Results / Summary

### LLM

The original deposit provides 54 merged activity summaries, 162 reported per-round coefficients, and raw dynamic traces for six chemically defined substrates. These quantitative measurements can be linked to their specific synthetic contexts, while proteomic occurrence counts remain a separate evidence type.

### Person

## Common numerical parameters

`common_parameters` contains numeric columns, including positional probabilities and their sum. PICS frequencies are occurrence counts divided by the observed total, separately by enzyme and library. QF relative activity follows the source normalization 100 × k/k_PLG; fluorescence slopes are refitted over the source-selected interval after averaging duplicate wells at each time. Source rounds and merged summaries remain identifiable. Missing Dpa probabilities are retained with explicit coverage. All transformations stay in memory; no CSV or database is written.

In [4]:
# Use the independent data library from this checkout.
import sys
sys.path.insert(0, str(ROOT / "assets/tools/proteolysis_data_manager/src"))
from proteolysis_data_manager.integration.merge import (
    build_dataset, parse_dataset, add_merops_scores,
)
contract = parse_dataset("mmp_pics_qf", ROOT / "data/proteolysis")
contract = add_merops_scores(
    contract, ROOT / "data/proteolysis/window_neighbourhood_datasets/raw/merops",
    dataset_index=34,
)
common_parameters = contract.to_frame()
preview = common_parameters.loc[
    common_parameters.relative_activity_percent.notna()
    & common_parameters.enzyme_name.eq("MMP1")
].head(6)
display(preview[[
    "enzyme_name", "substrate_name", "window", "source_k",
    "relative_activity_percent", "fluorescence_rate_au_s", "fluorescence_fit_r_squared",
    "position_probability_sum", "probability_positions_scored",
]])


,enzyme_name,substrate_name,window,source_k,relative_activity_percent,fluorescence_rate_au_s,fluorescence_fit_r_squared,position_probability_sum,probability_positions_scored
0,MMP1,Mca-PLG↓L-Dpa-AR-NH2,- P L G | L Dpa A R,1.128738,100.000000,1.126743,0.995723,1.308933,6
1,MMP1,Mca-PAN↓L-Dpa-AR-NH2,- P A N | L Dpa A R,0.673630,59.679887,0.672235,0.990315,1.030039,6
2,MMP1,Mca-PLN↓L-Dpa-AR-NH2,- P L N | L Dpa A R,2.851020,252.584779,2.850167,0.998365,1.104113,6
3,MMP1,Mca-PAG↓L-Dpa-AR-NH2,- P A G | L Dpa A R,0.121609,10.773856,0.122015,0.932062,1.234859,6
4,MMP1,Mca-VLG↓L-Dpa-AR-NH2,- V L G | L Dpa A R,0.207615,18.393568,0.204910,0.975158,1.031824,6
5,MMP1,Mca-VAN↓L-Dpa-AR-NH2,- V A N | L Dpa A R,0.803788,71.211238,0.802388,0.995275,0.752931,6
